In [1]:
import tensorflow as tf
import numpy as np
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Conv2DTranspose, BatchNormalization, Dropout, Lambda
from tensorflow.keras.optimizers import Adam
from keras.layers import Activation, MaxPool2D, Concatenate
from keras.losses import BinaryCrossentropy
from keras.metrics import MeanIoU
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from pathlib import Path
import os
import cv2
import imutils
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [2]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

# Dataset 

In [3]:
path=Path('../../../../Datasets/Processed/Saeedi')
path_tot=path/'Images'
path_gt=path/'GT_ICM'
path_models=Path('DATASET_KFOLD/ICM')

In [4]:
dim=(256,256)
def load_imgs(path, path_gt=''):
    files=[path/f for f in os.listdir(path)]
    x = np.array([cv2.resize(cv2.imread(str(f),cv2.IMREAD_GRAYSCALE), dim) for f in files])
    if path_gt == '':
        y=None
    else:
        y = np.array([cv2.threshold(cv2.resize(cv2.imread(str(path_gt/(f.stem+' ICM_Mask.bmp')),cv2.IMREAD_GRAYSCALE), dim),127,255,cv2.THRESH_BINARY)[1] for f in files])
    return x,y

In [5]:
x,y=load_imgs(path_tot,path_gt)

In [6]:
def std_norm(x):
    mean, std= np.mean(x, axis=0), np.std(x, axis=0) 
    return ((x.astype('float32') - mean) / std , mean, std)

In [7]:
def std_norm_test(x,mean,std):
    return (x.astype('float32') - mean) / std

In [8]:
def data_augmentation(x_train, y_train):
    x_train_augmented, y_train_augmented=[], []
    for k in range(len(x_train)):
        img=x_train[k]
        mask=y_train[k]
        for i in range(37):
            x_train_augmented.append(imutils.rotate(img, angle=10*i))
            y_train_augmented.append(imutils.rotate(mask, angle=10*i))
    return np.array(x_train_augmented), np.array(y_train_augmented)

In [9]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

# Model

In [10]:
def conv_block(input, num_filters):
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(input)
    x = Dropout(0.4) (x)
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(x)
    x = Dropout(0.4) (x)
    return x

#Encoder block: Conv block followed by maxpooling and residual block

def encoder_block(input, num_filters):
    x = conv_block(input, num_filters) # convolutional block
#     x = BatchNormalization()(x)
    input = Conv2D(num_filters, 1, padding="same")(input)
    transfer = tf.math.add(x,input)
    p = MaxPool2D(2)(transfer) #subsampling block  
    p = Activation("relu")(p)
    return x, p  


#Bottleneck block: 4 dilaten convolution layers

def bottleneck_block(input, num_filters):
    x1 = Conv2D(num_filters, 3, dilation_rate=1,  padding="same" )(input)
    x1 =  Dropout(0.4) (x1)
    x2 = Conv2D(num_filters, 3, dilation_rate=2, padding="same" )(x1)
    x2 =  Dropout(0.4) (x2)
    x3 = Conv2D(num_filters, 3, dilation_rate=4, padding="same" )(x2)
    x3 =  Dropout(0.4) (x3)
    x4 = Conv2D(num_filters, 3, dilation_rate=8, padding="same" )(x3)
    x4 =  Dropout(0.4) (x4)
    c = Concatenate(axis=3)([x1, x2, x3, x4])
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(c)
    x = Dropout(0.05) (x)
    return  x


#Decoder block
#skip features gets input from encoder for concatenation

def decoder_block(input, skip_features, num_filters):
    x = UpSampling2D(size=2)(input)
    x = BatchNormalization()(x) 
    x1 = Concatenate(axis=3)([x, skip_features])
      
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(x1)
    x = Dropout(0.4) (x)
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(x)
    x = Dropout(0.4) (x)
    x1 = Conv2D(num_filters, 1, padding="same")(x1)
    return tf.math.add(x,x1)

#Build Unet using the blocks
def build_unet(input_shape):
    inputs = Input(input_shape)

    s1, p1 = encoder_block(inputs, 16)
    s2, p2 = encoder_block(p1, 32)
    s3, p3 = encoder_block(p2, 64)
    s4, p4 = encoder_block(p3, 128)

    b1   = bottleneck_block(p4, 128) #Bridge

    d1 = decoder_block(b1, s4, 128)
    d2 = decoder_block(d1, s3, 64)
    d3 = decoder_block(d2, s2, 32)
    d4 = decoder_block(d3, s1, 16)

    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)  
    model = Model(inputs, outputs, name="U-Net")
    return model

In [11]:
model= build_unet(input_shape=(256,256,1))

In [12]:
def jaccard_index(y_true, y_pred):
    """ Calculates mean of Jaccard index as a loss function """
    intersection = tf.reduce_sum(y_true * y_pred, axis=(1,2))
    sum_ = tf.reduce_sum(y_true + y_pred, axis=(1,2))
    jac = (intersection) / (sum_ - intersection)
    return tf.reduce_mean(jac)

def my_loss_fn(y_true, y_pred):
    return BinaryCrossentropy(from_logits=True)(y_true, y_pred)-tf.math.log(jaccard_index(y_true, y_pred))

In [13]:
callbacks = [
    EarlyStopping(patience=15),
    ReduceLROnPlateau(factor=0.05, patience=5)]

In [14]:
from tensorflow.keras.utils import Sequence
class DataGenerator(Sequence):
    def __init__(self, x_set, y_set, batch_size):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.x) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_x = self.x[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_y = self.y[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x, batch_y=shuffle(batch_x, batch_y, random_state=1)
        return batch_x, batch_y

In [24]:
def accuracy(target, prediction):
    true_detec = np.logical_not(np.logical_xor(target, prediction))
    return np.sum(true_detec)/np.sum(np.ones_like(target))

def precision(target, prediction):
    intersection = np.logical_and(target, prediction)
    return np.sum(intersection)/(np.sum(prediction)+1)

def recall(target, prediction):
    intersection = np.logical_and(target, prediction)
    return np.sum(intersection)/np.sum(target)

def jaccard(target, prediction):
    intersection = np.logical_and(target, prediction)
    union = np.logical_or(target, prediction)
    return np.sum(intersection) / np.sum(union)

def dice(target, prediction):
    intersection = np.logical_and(target, prediction)
    return 2*np.sum(intersection) / (np.sum(target) + np.sum(prediction))

def metrics(target, prediction):
    return {'accuracy': accuracy(target, prediction),
            'precision': precision(target, prediction),
            'recall': recall(target, prediction),
            'specificity': recall(1-target,1- prediction),
            'jaccard':jaccard(target, prediction),
            'dice': dice(target, prediction)}

def summary_metrics(y_test,predictions,thresh=0.5):
    a,p,r,s,j,d=0.,0.,0.,0.,0.,0.
    n=len(predictions)
    for i in range(n):
        preds= (predictions[i][:,:,0]>=thresh).astype('uint8')
        gt= y_test[i].astype('uint8')
        metricas=metrics(gt,preds)
        a+=metricas['accuracy']
        p+=metricas['precision']
        r+=metricas['recall']
        s+=metricas['specificity']
        j+=metricas['jaccard']
        d+=metricas['dice']
    return {'accuracy': a/n,
            'precision': p/n,
            'recall': r/n,
            'specificity': s/n,
            'jaccard':j/n,
            'dice': d/n}

In [29]:
fold_no = 1
for train, test in kfold.split(x):
    x_train, y_train = x[train], y[train]
    x_test, y_test = x[test], y[test]
    
    x_train, mean, std = std_norm(x_train)
    x_test = std_norm_test(x_test, mean, std)
    
    x_train, y_train = data_augmentation(x_train, y_train)
    x_train, y_train = shuffle(x_train, y_train, random_state=13)
    
    x_train = x_train.astype("float32") 
    y_train = y_train.astype("float32")/255  

    x_test = x_test.astype("float32") 
    y_test = y_test.astype("float32")/255  
    
    train_gen = DataGenerator(x_train, y_train, 16)
    model= build_unet(input_shape=(256,256,1))
    model.compile(optimizer=Adam(learning_rate=0.0001), loss=my_loss_fn, metrics=[jaccard_index])
    
    print(f'Training for fold {fold_no} ...')
    history = model.fit(train_gen,batch_size=16, epochs=200, callbacks=callbacks, validation_data=(x_test, y_test))
    model.save('fold'+str(fold_no))    
    fold_no+= 1

Training for fold 1 ...
Epoch 1/200
518/518 [==============================] - 111s 172ms/step - loss: 1.4248 - jaccard_index: 0.5325 - val_loss: 1.1304 - val_jaccard_index: 0.6314
Epoch 2/200
518/518 [==============================] - 86s 166ms/step - loss: 1.0288 - jaccard_index: 0.7044 - val_loss: 1.0492 - val_jaccard_index: 0.6809
Epoch 3/200
518/518 [==============================] - 85s 165ms/step - loss: 0.9828 - jaccard_index: 0.7360 - val_loss: 1.0414 - val_jaccard_index: 0.6854
Epoch 4/200
518/518 [==============================] - 87s 167ms/step - loss: 0.9578 - jaccard_index: 0.7539 - val_loss: 1.0237 - val_jaccard_index: 0.7037
Epoch 5/200
518/518 [==============================] - 3763s 7s/step - loss: 0.9403 - jaccard_index: 0.7666 - val_loss: 0.9796 - val_jaccard_index: 0.7325
Epoch 6/200
518/518 [==============================] - 85s 164ms/step - loss: 0.9272 - jaccard_index: 0.7764 - val_loss: 0.9945 - val_jaccard_index: 0.7195
Epoch 7/200
518/518 [===================

Epoch 35/200
518/518 [==============================] - 86s 166ms/step - loss: 0.8285 - jaccard_index: 0.8532 - val_loss: 0.8996 - val_jaccard_index: 0.7962
Epoch 36/200
518/518 [==============================] - 86s 166ms/step - loss: 0.8288 - jaccard_index: 0.8530 - val_loss: 0.8992 - val_jaccard_index: 0.7966
Epoch 37/200
518/518 [==============================] - 86s 166ms/step - loss: 0.8280 - jaccard_index: 0.8536 - val_loss: 0.8983 - val_jaccard_index: 0.7976
Epoch 38/200
518/518 [==============================] - 86s 167ms/step - loss: 0.8282 - jaccard_index: 0.8534 - val_loss: 0.8992 - val_jaccard_index: 0.7967
Epoch 39/200
518/518 [==============================] - 86s 166ms/step - loss: 0.8280 - jaccard_index: 0.8536 - val_loss: 0.8998 - val_jaccard_index: 0.7961
INFO:tensorflow:Assets written to: fold4\assets
Training for fold 5 ...
Epoch 1/200
518/518 [==============================] - 91s 167ms/step - loss: 1.4439 - jaccard_index: 0.5237 - val_loss: 1.0790 - val_jaccard_i

# Metrics

In [25]:
fold_no = 1
m1=[]
for train, test in kfold.split(x):
    x_train, y_train = x[train], y[train]
    x_test, y_test = x[test], y[test]
    
    x_train, mean, std = std_norm(x_train)
    x_test = std_norm_test(x_test, mean, std)

    x_test = x_test.astype("float32") 
    y_test = y_test.astype("float32")/255 
    
    model= tf.keras.models.load_model(path_models/'fold{}'.format(fold_no),custom_objects={'jaccard_index':jaccard_index, 'my_loss_fn':my_loss_fn})
    predictions = model.predict(x_test)
    m1.append(summary_metrics(y_test,predictions, 0.5))
    fold_no = fold_no + 1

In [26]:
accuracy=[el['accuracy'] for el in m1]
precision=[el['precision'] for el in m1]
recall=[el['recall'] for el in m1]
specificity=[el['specificity'] for el in m1]
jaccard=[el['jaccard'] for el in m1]
dice=[el['dice'] for el in m1]

In [27]:
np.mean(accuracy), np.std(accuracy)

(0.974329111735026, 0.005597897589258854)

In [28]:
np.mean(precision), np.std(precision)

(0.8687725468692218, 0.03545228924984853)

In [20]:
np.mean(recall), np.std(recall)

(0.7425914060851972, 0.08156253881899679)

In [21]:
np.mean(specificity), np.std(specificity)

(0.9936271641360397, 0.0018805912737353532)

In [22]:
np.mean(jaccard), np.std(jaccard)

(0.6820147177855528, 0.07367079131200896)

In [23]:
np.mean(dice), np.std(dice)

(0.7738471012491568, 0.06801150200670072)